# Weekday Pattern **Baseline Only** (Option B) — No Clustering

You chose **Option B**: **do not cluster** because the weekday-only signal does not produce stable, meaningful multi-cluster structure under your filters.

What this notebook does instead:
- Aggregates to **daily** quantities per product (across stores).
- Removes **day-level spikes** per product (robust MAD rule) so weekday means are not distorted.
- Computes **weekday share vector** (Mon..Sun) per product (level-invariant).
- Computes **data-driven outliers** (quantile-based sparsity + optional IsolationForest).
- Produces a **single baseline weekday profile** for all *inlier* products (and optional category-level baselines).

## Inputs (expected in `raw_data/`)
- `20260218_144523_sales_data.parquet`
- `20260218_144523_holidays.parquet`

## Outputs (written to `reports/`)
- `run_config_baseline.json`
- `weekday_baseline_inliers.csv`
- `weekday_baseline_by_category_inliers.csv`
- `product_weekday_shares.parquet`
- `product_outliers.csv`


In [10]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import IsolationForest

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)


## 1) Paths / Parameters

In [11]:
# Repo root detection (works even if notebook is opened inside src/feature_engineering)
HERE = Path.cwd()
if HERE.name == "feature_engineering" and HERE.parent.name == "src":
    PROJECT_ROOT = HERE.parent.parent
elif HERE.name == "src":
    PROJECT_ROOT = HERE.parent
else:
    PROJECT_ROOT = HERE

RAW_DIR = PROJECT_ROOT / "raw_data"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SALES_FILE = RAW_DIR / "20260218_144523_sales_data.parquet"
HOLIDAYS_FILE = RAW_DIR / "20260218_144523_holidays.parquet"

# --- Core decisions ---
EXCLUDE_HOLIDAYS = True          # Remove holidays from weekday means (default True)
DAY_OUTLIER_K = 6.0              # Robust spike filter (MAD z-score)
LOOKBACK_DAYS = 56               # Window used ONLY for outlier stability metrics

# --- Outlier thresholds (quantile-based, data-driven) ---
# These control how many products are excluded as "weekday profile unreliable"
Q_MIN_ACTIVE_DAYS = 0.20         # higher => stricter (more outliers)
Q_MAX_ZERO_RATIO = 0.80          # lower => stricter (more outliers)

# Optional anomaly model on weekday share vectors (7D)
USE_IFOREST = True
Q_CONTAMINATION_TAIL = 0.98      # top 2% flagged as anomalies

RANDOM_STATE = 42

print("CWD:", HERE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SALES_FILE exists:", SALES_FILE.exists())
print("HOLIDAYS_FILE exists:", HOLIDAYS_FILE.exists())

cfg = {
    "MODE": "BASELINE_ONLY_NO_CLUSTERING",
    "EXCLUDE_HOLIDAYS": EXCLUDE_HOLIDAYS,
    "DAY_OUTLIER_K": DAY_OUTLIER_K,
    "LOOKBACK_DAYS": LOOKBACK_DAYS,
    "Q_MIN_ACTIVE_DAYS": Q_MIN_ACTIVE_DAYS,
    "Q_MAX_ZERO_RATIO": Q_MAX_ZERO_RATIO,
    "USE_IFOREST": USE_IFOREST,
    "Q_CONTAMINATION_TAIL": Q_CONTAMINATION_TAIL,
    "RANDOM_STATE": RANDOM_STATE,
}
(REPORTS_DIR / "run_config_baseline.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
cfg


CWD: c:\Users\simon\food_prediction\src\feature_engineering
PROJECT_ROOT: c:\Users\simon\food_prediction
SALES_FILE exists: True
HOLIDAYS_FILE exists: True


{'MODE': 'BASELINE_ONLY_NO_CLUSTERING',
 'EXCLUDE_HOLIDAYS': True,
 'DAY_OUTLIER_K': 6.0,
 'LOOKBACK_DAYS': 56,
 'Q_MIN_ACTIVE_DAYS': 0.2,
 'Q_MAX_ZERO_RATIO': 0.8,
 'USE_IFOREST': True,
 'Q_CONTAMINATION_TAIL': 0.98,
 'RANDOM_STATE': 42}

## 2) Load data

In [12]:
sales = pd.read_parquet(SALES_FILE)
holidays = pd.read_parquet(HOLIDAYS_FILE)

print("sales columns:", list(sales.columns))
print("holidays columns:", list(holidays.columns))
sales.head()


sales columns: ['date', 'category_name', 'item_id', 'sold_quantity', 'price', 'store_id']
holidays columns: ['zipcode', 'subdivision_code', 'date', 'holiday_name', 'holiday_type']


,date,category_name,item_id,sold_quantity,price,store_id
0,2025-04-01,Angebot Brötchen,139,15.0,0.0,0
1,2025-04-01,Angebot Feinbäckerei,138,28.0,0.0,0
2,2025-04-01,Angebot Heißgetränke,106,25.0,0.0,0
3,2025-04-01,Angebot Heißgetränke,539,5.0,1.4,0
4,2025-04-01,Angebot Snack,176,1.0,0.0,0


## 3) Helpers

In [13]:
def safe_to_datetime(s: pd.Series) -> pd.Series:
    """Parse to datetime and normalize to midnight (daily granularity)."""
    return pd.to_datetime(s, errors="coerce").dt.normalize()


def aggregate_daily_product_sales(sales_df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate to daily totals per product (across stores).

    Expected schema:
      ['date','category_name','item_id','sold_quantity','price','store_id']

    Output columns:
      ['date','product_id','category_name','qty']
    """
    df = sales_df.copy()
    df["date"] = safe_to_datetime(df["date"])
    df = df.dropna(subset=["date", "item_id"])
    df["sold_quantity"] = pd.to_numeric(df["sold_quantity"], errors="coerce").fillna(0.0)

    daily = (
        df.groupby(["date", "item_id", "category_name"], as_index=False)["sold_quantity"]
        .sum()
        .rename(columns={"item_id": "product_id", "sold_quantity": "qty"})
    )
    return daily


def build_holiday_set(holidays_df: pd.DataFrame) -> set[pd.Timestamp]:
    """Return holiday dates as a set for fast filtering."""
    date_col = None
    for c in ["date", "ds", "day", "timestamp", "datetime"]:
        if c in holidays_df.columns:
            date_col = c
            break
    if date_col is None:
        return set()
    return set(safe_to_datetime(holidays_df[date_col]).dropna().unique())


def remove_outlier_days_mad(daily: pd.DataFrame, k: float = 6.0) -> pd.DataFrame:
    """
    Remove per-product day spikes using robust MAD z-score.
    Outlier day qty is set to NaN so it does not affect weekday means.
    """
    df = daily.copy()

    med = df.groupby("product_id")["qty"].transform("median")

    def _mad(s: pd.Series) -> float:
        m = float(np.median(s))
        return float(np.median(np.abs(s - m)))

    mad = df.groupby("product_id")["qty"].transform(_mad)
    scale = 1.4826 * mad.replace(0, np.nan)

    z = (df["qty"] - med) / scale
    df["is_day_outlier"] = z.abs() > k

    df.loc[df["is_day_outlier"] == True, "qty"] = np.nan
    return df


## 4) Compute per-product weekday shares (level-invariant)

In [14]:
holiday_set = build_holiday_set(holidays)

daily = aggregate_daily_product_sales(sales)
daily = remove_outlier_days_mad(daily, k=DAY_OUTLIER_K)

daily["date"] = safe_to_datetime(daily["date"])
daily["dow"] = daily["date"].dt.dayofweek  # 0=Mon .. 6=Sun

# For weekday profiles, optionally exclude holiday dates (they behave like special days)
profile_df = daily.copy()
if EXCLUDE_HOLIDAYS and holiday_set:
    profile_df = profile_df.loc[~profile_df["date"].isin(holiday_set)].copy()

# Compute mean qty per weekday per product
w = (
    profile_df.groupby(["product_id", "dow"], as_index=False)["qty"]
    .mean(numeric_only=True)
    .pivot(index="product_id", columns="dow", values="qty")
    .fillna(0.0)
).reindex(columns=list(range(7)), fill_value=0.0)

# Normalize to shares (sum to 1) => magnitude removed
wsum = w.sum(axis=1).replace(0, np.nan)
weekday_shares = w.div(wsum, axis=0).fillna(0.0)
weekday_shares.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

weekday_shares.head()


,Mon,Tue,Wed,Thu,Fri,Sat,Sun
product_id,,,,,,,
0,0.118747,0.112920,0.119435,0.117695,0.140550,0.209355,0.181297
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.108076,0.103953,0.115450,0.107113,0.130598,0.233627,0.201183
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000446,0.000668,0.000891,0.000000,0.000891,0.503843,0.493261


## 5) Outlier detection (data-driven)

In [15]:
# We exclude products with unreliable weekday shares:
# - too few active days in the last LOOKBACK_DAYS
# - too many zero days in the last LOOKBACK_DAYS
# plus optional anomaly detection in weekday-share space (7D).

max_date = daily["date"].max()
start_date = max_date - pd.Timedelta(days=LOOKBACK_DAYS - 1)
window_dates = pd.date_range(start_date, max_date, freq="D")

win = daily.loc[(daily["date"] >= start_date) & (daily["date"] <= max_date)].copy()

X = (
    win.pivot_table(index="product_id", columns="date", values="qty", aggfunc="sum")
    .reindex(columns=window_dates, fill_value=0.0)
    .fillna(0.0)
)

meta = pd.DataFrame(index=X.index)
meta["active_days_window"] = (X.values > 0).sum(axis=1)
meta["zero_ratio_window"] = 1.0 - meta["active_days_window"] / float(LOOKBACK_DAYS)

# Quantile thresholds
min_active_days = int(np.quantile(meta["active_days_window"].values, Q_MIN_ACTIVE_DAYS))
min_active_days = max(7, min_active_days)

max_zero_ratio = float(np.quantile(meta["zero_ratio_window"].values, Q_MAX_ZERO_RATIO))
max_zero_ratio = float(np.clip(max_zero_ratio, 0.0, 0.98))

contamination = float(1.0 - Q_CONTAMINATION_TAIL)
contamination = float(np.clip(contamination, 0.005, 0.10))

print("min_active_days:", min_active_days)
print("max_zero_ratio:", max_zero_ratio)
print("iforest contamination:", contamination)

out = meta.copy()
out["rule_sparse"] = (out["active_days_window"] < min_active_days) | (out["zero_ratio_window"] > max_zero_ratio)

# Align share vectors to meta index
W = weekday_shares.reindex(out.index).fillna(0.0).values

out["iforest_outlier"] = False
if USE_IFOREST:
    iso = IsolationForest(
        n_estimators=400,
        contamination=contamination,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    out["iforest_outlier"] = (iso.fit_predict(W) == -1)

out["is_outlier_product"] = out["rule_sparse"] | out["iforest_outlier"]
out["outlier_reason"] = ""
out.loc[out["rule_sparse"], "outlier_reason"] = "sparse_or_intermittent"
out.loc[(~out["rule_sparse"]) & (out["iforest_outlier"]), "outlier_reason"] = "anomalous_weekday_profile"

out["is_outlier_product"].value_counts()


min_active_days: 7
max_zero_ratio: 0.9642857142857143
iforest contamination: 0.020000000000000018


is_outlier_product
False    439
True     153
Name: count, dtype: int64

## 6) Baseline weekday profile for inliers

In [16]:
# Baseline: mean of weekday-share vectors across inlier products.
# This gives you ONE stable "expected weekday distribution" (ignoring magnitude).

inlier_ids = out.index[~out["is_outlier_product"]].tolist()

baseline_vec = weekday_shares.loc[inlier_ids].mean(axis=0)
baseline = baseline_vec.to_frame(name="share").reset_index(names="weekday")

baseline


,weekday,share
0,Mon,0.144956
1,Tue,0.138515
2,Wed,0.146956
3,Thu,0.150035
4,Fri,0.161833
5,Sat,0.168977
6,Sun,0.088729


## 7) Optional: baseline by category (inliers only)

In [17]:
# Map each product_id to a category_name.
# If a product appears in multiple categories (shouldn't, but can happen),
# we pick the most frequent category assignment.

prod_cat = (
    sales[["item_id", "category_name"]]
    .rename(columns={"item_id": "product_id"})
    .dropna()
)

prod_cat_mode = (
    prod_cat.groupby("product_id")["category_name"]
    .agg(lambda s: s.value_counts().index[0])
    .rename("category_name")
)

shares_with_cat = weekday_shares.merge(prod_cat_mode.to_frame(), left_index=True, right_index=True, how="left")

inlier_shares_with_cat = shares_with_cat.loc[inlier_ids].copy()

by_cat = (
    inlier_shares_with_cat
    .groupby("category_name")[["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]]
    .mean(numeric_only=True)
    .reset_index()
)

by_cat.head(10)


,category_name,Mon,Tue,Wed,Thu,Fri,Sat,Sun
0,Angebot Brot,0.145145,0.082946,0.325197,0.164905,0.125515,0.116936,0.039355
1,Angebot Brötchen,0.135150,0.129439,0.131454,0.158037,0.169966,0.173361,0.102594
2,Angebot Feinbäckerei,0.157491,0.121282,0.176703,0.163734,0.170381,0.130672,0.079737
3,Angebot Heißgetränke,0.107217,0.140634,0.151082,0.153617,0.212105,0.169151,0.066194
4,Angebot Kuchen,0.124869,0.088143,0.144806,0.190976,0.132214,0.245540,0.073452
5,Angebot Snack,0.173309,0.184974,0.169253,0.195678,0.132603,0.070826,0.073357
6,Brot,0.133963,0.125415,0.148304,0.133371,0.170993,0.237215,0.050740
7,Brötchen,0.112721,0.113577,0.121562,0.115364,0.147656,0.237350,0.151770
8,Eigerichte,0.155473,0.144593,0.152199,0.144209,0.140183,0.163805,0.099537
9,Feinbäckerei,0.144256,0.131954,0.133419,0.185746,0.195887,0.157094,0.051644


## 8) Save outputs

In [18]:
# Save per-product weekday shares (so you can join them into your model features later)
shares_out = weekday_shares.copy()
shares_out.index.name = "product_id"
shares_out = shares_out.reset_index()

# Attach outlier flags
out_out = out.reset_index(names="product_id")

shares_out = shares_out.merge(out_out[["product_id", "is_outlier_product", "outlier_reason"]], on="product_id", how="left")

shares_out.to_parquet(REPORTS_DIR / "product_weekday_shares.parquet", index=False)

baseline.to_csv(REPORTS_DIR / "weekday_baseline_inliers.csv", index=False)
by_cat.to_csv(REPORTS_DIR / "weekday_baseline_by_category_inliers.csv", index=False)

# Outlier list
outliers = out_out.loc[out_out["is_outlier_product"] == True, ["product_id", "rule_sparse", "iforest_outlier", "outlier_reason", "active_days_window", "zero_ratio_window"]]
outliers.to_csv(REPORTS_DIR / "product_outliers.csv", index=False)

print("Wrote:")
print(" -", REPORTS_DIR / "run_config_baseline.json")
print(" -", REPORTS_DIR / "weekday_baseline_inliers.csv")
print(" -", REPORTS_DIR / "weekday_baseline_by_category_inliers.csv")
print(" -", REPORTS_DIR / "product_weekday_shares.parquet")
print(" -", REPORTS_DIR / "product_outliers.csv")

baseline


Wrote:
 - c:\Users\simon\food_prediction\reports\run_config_baseline.json
 - c:\Users\simon\food_prediction\reports\weekday_baseline_inliers.csv
 - c:\Users\simon\food_prediction\reports\weekday_baseline_by_category_inliers.csv
 - c:\Users\simon\food_prediction\reports\product_weekday_shares.parquet
 - c:\Users\simon\food_prediction\reports\product_outliers.csv


,weekday,share
0,Mon,0.144956
1,Tue,0.138515
2,Wed,0.146956
3,Thu,0.150035
4,Fri,0.161833
5,Sat,0.168977
6,Sun,0.088729
